# Training Sentiment Analysis

Melatih 3 model untuk klasifikasi sentimen ulasan Bank Jago.

In [ ]:
import os, json, joblib, re, warnings
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

warnings.filterwarnings('ignore')
SEED = 42
np.random.seed(SEED)

In [ ]:
# Load & label
SENTIMENT_LABELS = ['Negative', 'Neutral', 'Positive']
def rating_to_label(score):
    if score <= 2: return 0
    if score == 3: return 1
    return 2

def clean_text(text):
    text = str(text).lower()
    text = re.sub(r'https?://\\S+|www\\.\\S+', '', text)
    text = re.sub(r'<.*?>+', '', text)
    text = re.sub(r'[^\w\s]', '', text)
    text = re.sub(r'\d+', '', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

df = pd.read_csv('data/raw/reviews.csv')
df['label'] = df['score'].apply(rating_to_label)
df['clean'] = df['content'].apply(clean_text)
df = df[df['clean'].str.len() > 0].reset_index(drop=True)
print(f'Total samples: {len(df)}')
print(f'Label distribution:\n{df["label"].value_counts().sort_index()}')

In [ ]:
# Train/test split
X = df['clean'].values
y = df['label'].values
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=SEED)
print(f'Train: {len(X_train)}, Test: {len(X_test)}')

## Experiment 1: TF-IDF + Logistic Regression

In [ ]:
vec1 = TfidfVectorizer(max_features=10000, ngram_range=(1, 2), sublinear_tf=True)
X_train_1 = vec1.fit_transform(X_train)
X_test_1 = vec1.transform(X_test)

model1 = LogisticRegression(max_iter=1000, class_weight='balanced', random_state=SEED, n_jobs=-1)
model1.fit(X_train_1, y_train)
y_pred1 = model1.predict(X_test_1)

acc1 = accuracy_score(y_test, y_pred1)
print(f'Accuracy: {acc1:.2%}')
print(classification_report(y_test, y_pred1, target_names=SENTIMENT_LABELS, zero_division=0))

## Experiment 2: TF-IDF + Linear SVM

In [ ]:
vec2 = TfidfVectorizer(max_features=15000, ngram_range=(1, 2), sublinear_tf=True)
X_train_2 = vec2.fit_transform(X_train)
X_test_2 = vec2.transform(X_test)

model2 = LinearSVC(max_iter=2000, class_weight='balanced', random_state=SEED)
model2.fit(X_train_2, y_train)
y_pred2 = model2.predict(X_test_2)

acc2 = accuracy_score(y_test, y_pred2)
print(f'Accuracy: {acc2:.2%}')
print(classification_report(y_test, y_pred2, target_names=SENTIMENT_LABELS, zero_division=0))

## Experiment 3: Bag-of-Words + Logistic Regression

In [ ]:
vec3 = TfidfVectorizer(max_features=8000, ngram_range=(1, 1), sublinear_tf=True)
X_train_3 = vec3.fit_transform(X_train)
X_test_3 = vec3.transform(X_test)

model3 = LogisticRegression(max_iter=1000, class_weight='balanced', random_state=SEED, n_jobs=-1)
model3.fit(X_train_3, y_train)
y_pred3 = model3.predict(X_test_3)

acc3 = accuracy_score(y_test, y_pred3)
print(f'Accuracy: {acc3:.2%}')
print(classification_report(y_test, y_pred3, target_names=SENTIMENT_LABELS, zero_division=0))

## Summary

In [ ]:
print(f'EXP-01 (LR+TF-IDF): {acc1:.2%}')
print(f'EXP-02 (SVM+TF-IDF): {acc2:.2%}')
print(f'EXP-03 (LR+BoW):     {acc3:.2%}')
print(f'Best model: {"EXP-02 (SVM)" if max(acc1,acc2,acc3)==acc2 else "EXP-01 (LR)" if max(acc1,acc2,acc3)==acc1 else "EXP-03 (LR BoW)"} with {max(acc1,acc2,acc3):.2%} accuracy')